<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.9em; font-weight:bold; margin:0.6em 0 0.3em 0;">מחברת N2.4: חקר נתונים (EDA) קצר על מאגר חדש</p>
<p dir="rtl" style="direction:rtl; text-align:right;">משך מוערך: 15 דקות. אתם עוברים על אותו מסלול של חלק ד בשיעור, הפעם על מאגר נתונים אחר: ספירה וסיכום לפי קבוצה, צורת ההתפלגות וחריגים, שלושה סוגי שונות משותפת, והשערה אחת.</p>
<p dir="rtl" style="direction:rtl; text-align:right;"><b>תא בדיקה</b> מדפיס <code dir="ltr">OK</code> כשהתשובה נכונה; אם מופיעה שגיאה, תקנו והריצו שוב. עבדו לפי הסדר: קראו, השלימו את החסר (<code dir="ltr">___</code>), הריצו, בדקו.</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנת הסביבה</p>
<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא מתקין רק את החבילות הדרושות, ממאגר בינארי מהיר, ומדפיס כמה זמן ההתקנה נמשכה. הריצה הראשונה אורכת זמן מה.</p>

In [ ]:
# Install only the packages the course needs, from a binary repository (fast)
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
needed  <- c("tidyverse")          # add packages here when a notebook needs them
missing <- setdiff(needed, rownames(installed.packages()))   # skip installed ones
t <- system.time(if (length(missing) > 0) install.packages(missing))
cat("Installed:", if (length(missing)) paste(missing, collapse = ", ") else "nothing new",
    "| elapsed seconds:", round(t[["elapsed"]], 1), "\n")
library(tidyverse)
theme_set(theme_minimal(base_size = 14))   # one clean theme for all graphs

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">המאגר ושני מבחנים חדשים: תזכורת</p>
<p dir="rtl" style="direction:rtl; text-align:right;">41 מטופלים אחרי שבץ ראשון (<span dir="ltr">Frenkel-Toledo et al., 2022, Scientific Reports 12:10169</span>, רשיון <span dir="ltr">CC BY 4.0</span>; <a href="https://doi.org/10.1038/s41598-022-14359-8">למאמר המקורי</a>). העמודות: <code dir="ltr">subject</code> (מספר נבדק), <code dir="ltr">sex</code>, <code dir="ltr">age</code>, <code dir="ltr">dominance</code> (יד דומיננטית), <code dir="ltr">lesion_site</code>, <code dir="ltr">lesion_type</code> (<span dir="ltr">I</span> = איסכמי, <span dir="ltr">H</span> = המורגי), <code dir="ltr">lesion_vol</code> (סמ"ק), <code dir="ltr">tao_months</code> (חודשים מההתחלה), <code dir="ltr">mas</code>, <code dir="ltr">fma</code> (<span dir="ltr">FMA-UE</span>, תפקוד מוטורי של הגף העליון, 0 עד 66) ו-<code dir="ltr">tsrt</code>. שני מבחנים חדשים לכם, לפני שמסתכלים בנתונים:</p>
<table dir="rtl" style="direction:rtl; text-align:right; border-collapse:collapse; width:100%;">
<tr style="background:#e8f0f0;"><th style="border:1px solid #999; padding:6px;"></th><th style="border:1px solid #999; padding:6px;"><span dir="ltr">MAS</span> (<span dir="ltr">Modified Ashworth Scale</span>)</th><th style="border:1px solid #999; padding:6px;"><span dir="ltr">TSRT</span> (<span dir="ltr">Tonic Stretch Reflex Threshold</span>)</th></tr>
<tr><td style="border:1px solid #999; padding:6px;"><b>מה מודד</b></td><td style="border:1px solid #999; padding:6px;">עלייה בטונוס השרירי: ההתנגדות של שריר להזזה פסיבית של המפרק (דירוג קליני של ספסטיות)</td><td style="border:1px solid #999; padding:6px;">הסף הזוויתי של רפלקס המתיחה בשריר רפוי: באיזו זווית מרפק מתחילה תגובה רפלקסיבית של השריר</td></tr>
<tr><td style="border:1px solid #999; padding:6px;"><b>איך</b></td><td style="border:1px solid #999; padding:6px;">הבודק מזיז את המפרק בהזזה פסיבית ומדרג את ההתנגדות שהוא מרגיש</td><td style="border:1px solid #999; padding:6px;">מותחים את השריר הרפוי במהירויות שונות, מזהים (לפי <span dir="ltr">EMG</span>) את הזווית שבה מתחילה פעילות, ומחשבים בקירוב ליניארי את הזווית במהירות אפס</td></tr>
<tr><td style="border:1px solid #999; padding:6px;"><b>טווח</b></td><td style="border:1px solid #999; padding:6px;">0, 1, 1+, 2, 3, 4 (סולם אורדינלי); גבוה = ספסטיות חמורה יותר. בקובץ שלנו: מכופפי המרפק, ו-1+ מקודד כ-<code dir="ltr">1.5</code></td><td style="border:1px solid #999; padding:6px;">מעלות. בקובץ שלנו: 62 עד 163. <b>זווית קטנה יותר = ספסטיות חמורה יותר</b></td></tr>
<tr><td style="border:1px solid #999; padding:6px;"><b>קריאה נוספת</b></td><td style="border:1px solid #999; padding:6px;"><span dir="ltr">sralab.org/rehabilitation-measures/ashworth-scale-modified-ashworth-scale</span> · <span dir="ltr">PubMed 3809245</span></td><td style="border:1px solid #999; padding:6px;"><span dir="ltr">PubMed 35715476</span> (המאמר שממנו הנתונים; סעיף השיטות)</td></tr>
</table>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">טעינת הנתונים</p>
<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא טוען את הטבלה <code dir="ltr">sp</code>. הפונקציה <code dir="ltr">glimpse()</code> מציגה את העמודות וסוגיהן.</p>

In [ ]:
source("https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/R/load_course_data.R")
sp <- load_course_data("spasticity_s1")   # one row per patient
glimpse(sp)

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.25em; font-weight:bold; margin:0.6em 0 0.3em 0;">1. כמה בכל קבוצה, ומה הסיכום לפי קבוצה</p>
<p dir="rtl" style="direction:rtl; text-align:right;"><b>א.</b> ספרו כמה מטופלים יש מכל סוג נגע (<code dir="ltr">lesion_type</code>). שימו לב: יש קבוצה של מטופל אחד בלבד.<br><b>ב.</b> צרו את הטבלה <code dir="ltr">sp2</code>, בלי קבוצת המטופל האחד, כלומר רק <code dir="ltr">"I"</code> ו-<code dir="ltr">"H"</code>.<br><b>ג.</b> חשבו לפי סוג נגע: מספר המטופלים (<code dir="ltr">n</code>) והחציון של <code dir="ltr">tsrt</code> ושל <code dir="ltr">fma</code>.</p>
<details dir="rtl" style="direction:rtl; text-align:right;"><summary><b>רמז</b></summary>
<pre dir="ltr" style="direction:ltr; text-align:left; background:#f4f4f4; padding:6px;"><code>count(sp, lesion_type)
filter(lesion_type %in% c("I", "H"))
group_by(lesion_type) |&gt; summarize(n = n(), median_tsrt = median(tsrt), median_fma = median(fma))</code></pre>
</details>

In [ ]:
sp |> count(___)                                  # patients per lesion type

In [ ]:
sp2 <- sp |> filter(lesion_type %in% c(___, ___))  # keep only the two sizeable groups

In [ ]:
by_type <- sp2 |>
  group_by(___) |>                                   # one group per lesion type
  summarize(n = n(),                                 # patients in the group
            median_tsrt = median(tsrt),              # median TSRT (degrees)
            median_fma = median(fma))                # median FMA-UE
by_type

In [ ]:
stopifnot(nrow(sp2) == 40, setequal(by_type$lesion_type, c("I", "H")),
          by_type$n[by_type$lesion_type == "I"] == 31,
          "median_tsrt" %in% names(by_type), "median_fma" %in% names(by_type))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.25em; font-weight:bold; margin:0.6em 0 0.3em 0;">2. צורת ההתפלגות וחריגים: נפח הנגע</p>
<p dir="rtl" style="direction:rtl; text-align:right;"><b>א.</b> ציירו היסטוגרמה של <code dir="ltr">lesion_vol</code> (נפח הנגע בסמ"ק) ב-<code dir="ltr">sp2</code>.<br><b>ב.</b> האם ההתפלגות מוטה שמאלה, מוטה ימינה או סימטרית? כתבו <code dir="ltr">"left"</code>, <code dir="ltr">"right"</code> או <code dir="ltr">"symmetric"</code>.<br><b>ג.</b> מצאו חריגים לפי כלל 1.5 מהטווח הבין-רבעוני (<span dir="ltr">IQR</span>): נפח גדול מהרבעון השלישי ועוד 1.5 פעמים ה-<span dir="ltr">IQR</span>. שמרו את שורותיהם ב-<code dir="ltr">outliers</code>.</p>
<details dir="rtl" style="direction:rtl; text-align:right;"><summary><b>רמז</b></summary>
<pre dir="ltr" style="direction:ltr; text-align:left; background:#f4f4f4; padding:6px;"><code>ggplot(sp2, aes(x = lesion_vol)) + geom_histogram(bins = 15)
filter(lesion_vol &gt; quantile(lesion_vol, 0.75) + 1.5 * IQR(lesion_vol))</code></pre>
</details>

In [ ]:
p_shape <- ggplot(sp2, aes(x = ___)) +               # one numeric column on x
  geom_histogram(bins = 15, fill = "#F58518", colour = "white") +   # 15 bars
  labs(x = "Lesion volume (cc)", y = "Patients")
p_shape

In [ ]:
lesion_vol_skew <- "___"      # "left", "right" or "symmetric"

In [ ]:
outliers <- sp2 |>
  filter(lesion_vol > quantile(lesion_vol, 0.75) + 1.5 * ___(lesion_vol))   # the 1.5 x IQR rule
outliers |> select(subject, lesion_type, lesion_vol, fma, tsrt)

In [ ]:
stopifnot(class(p_shape$layers[[1]]$geom)[1] == "GeomBar",
          rlang::as_label(p_shape$mapping$x) == "lesion_vol",
          lesion_vol_skew == "right",
          nrow(outliers) == 7)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>ד. שאלה למחשבה (בלי קוד):</b> האם שבעת החריגים הם טעויות הקלדה? מה הייתם בודקים לפני שמסירים אותם? (רמז: ראו בעמודות <code dir="ltr">fma</code> ו-<code dir="ltr">tsrt</code> של אותם מטופלים.)</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.25em; font-weight:bold; margin:0.6em 0 0.3em 0;">3. שונות משותפת (1): קטגוריה מול מספר</p>
<p dir="rtl" style="direction:rtl; text-align:right;">האם <code dir="ltr">tsrt</code> שונה בין נגע איסכמי להמורגי? ציירו תרשים קופסה של <code dir="ltr">tsrt</code> לפי <code dir="ltr">lesion_type</code>, והוסיפו את כל המטופלים כנקודות עם רעד (<code dir="ltr">outlier.shape = NA</code> מונע ציור כפול של חריגים). זכרו: <b>זווית קטנה = ספסטיות חמורה יותר.</b> בקבוצה אחת יש רק 9 מטופלים, ולכן הנקודות חשובות.</p>
<details dir="rtl" style="direction:rtl; text-align:right;"><summary><b>רמז</b></summary>
<pre dir="ltr" style="direction:ltr; text-align:left; background:#f4f4f4; padding:6px;"><code>ggplot(sp2, aes(x = lesion_type, y = tsrt)) +
  geom_boxplot(outlier.shape = NA) +
  geom_jitter(width = 0.12, height = 0)</code></pre>
</details>

In [ ]:
p_catnum <- ggplot(sp2, aes(x = ___, y = ___)) +     # category on x, number on y
  geom_boxplot(outlier.shape = NA) +                  # the box, without its own outlier points
  geom_jitter(width = 0.12, height = 0) +             # every patient as a point
  labs(x = "Lesion type (I = ischemic, H = hemorrhagic)", y = "TSRT (degrees)")
p_catnum

In [ ]:
stopifnot("GeomBoxplot" %in% sapply(p_catnum$layers, function(l) class(l$geom)[1]),
          rlang::as_label(p_catnum$mapping$x) == "lesion_type",
          rlang::as_label(p_catnum$mapping$y) == "tsrt")
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.25em; font-weight:bold; margin:0.6em 0 0.3em 0;">4. שונות משותפת (2): שתי קטגוריות</p>
<p dir="rtl" style="direction:rtl; text-align:right;">האם רמת הספסטיות הקלינית (<code dir="ltr">mas</code>) שונה בין שני סוגי הנגע? <b>א.</b> בנו טבלת שכיחויות <code dir="ltr">mas_tab</code> עם <code dir="ltr">count()</code> של שתי העמודות.<br><b>ב.</b> ציירו עמודות מוערמות שכל עמודה בהן מגיעה ל-100% (<code dir="ltr">position = "fill"</code>), כדי להשוות <b>אחוזים בתוך כל סוג</b> ולא מספרים. הפונקציה <code dir="ltr">factor(mas)</code> הופכת את הדרגות לקטגוריות כדי שהצבעים יהיו נפרדים.</p>
<details dir="rtl" style="direction:rtl; text-align:right;"><summary><b>רמז</b></summary>
<pre dir="ltr" style="direction:ltr; text-align:left; background:#f4f4f4; padding:6px;"><code>count(lesion_type, mas)
geom_bar(position = "fill")</code></pre>
</details>

In [ ]:
mas_tab <- sp2 |> count(___, mas)                    # frequency table: stroke type x MAS grade
mas_tab

In [ ]:
p_catcat <- ggplot(sp2, aes(x = lesion_type, fill = factor(mas))) +
  geom_bar(position = "___") +                       # every bar scaled to 100 percent
  labs(x = "Lesion type", y = "Share of patients", fill = "MAS (1.5 = 1+)")
p_catcat

In [ ]:
stopifnot(sum(mas_tab$n) == 40, all(c("lesion_type", "mas") %in% names(mas_tab)),
          class(p_catcat$layers[[1]]$position)[1] == "PositionFill")
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>ג. תרגיל קטן: שלוש דרכים לערום עמודות.</b> הפרמטר <code dir="ltr">position</code> של <code dir="ltr">geom_bar()</code> קובע איך עמודות של אותה קטגוריה מסודרות. הריצו את שלושת התאים הבאים, כשבכל אחד מהם מוכנס אחד משלושה ערכים: <code dir="ltr">"stack"</code> (ערימה: הברירת מחדל), <code dir="ltr">"dodge"</code> (עמודות זו ליד זו), <code dir="ltr">"fill"</code> (ערימה שמגיעה ל-100%).</p>
<details dir="rtl" style="direction:rtl; text-align:right;"><summary><b>רמז</b></summary>
<pre dir="ltr" style="direction:ltr; text-align:left; background:#f4f4f4; padding:6px;"><code>geom_bar(position = "stack")
geom_bar(position = "dodge")
geom_bar(position = "fill")</code></pre>
</details>

In [ ]:
p_stack <- ggplot(sp2, aes(x = lesion_type, fill = factor(mas))) +
  geom_bar(position = "___") +                       # TODO: "stack"
  labs(title = "stack", x = "Lesion type", y = "Patients", fill = "MAS")
p_stack

In [ ]:
p_dodge <- ggplot(sp2, aes(x = lesion_type, fill = factor(mas))) +
  geom_bar(position = "___") +                       # TODO: "dodge"
  labs(title = "dodge", x = "Lesion type", y = "Patients", fill = "MAS")
p_dodge

In [ ]:
p_fill <- ggplot(sp2, aes(x = lesion_type, fill = factor(mas))) +
  geom_bar(position = "___") +                       # TODO: "fill"
  labs(title = "fill", x = "Lesion type", y = "Share of patients", fill = "MAS")
p_fill

In [ ]:
pos <- function(p) class(p$layers[[1]]$position)[1]   # the position class of the first layer
stopifnot(pos(p_stack) == "PositionStack", pos(p_dodge) == "PositionDodge", pos(p_fill) == "PositionFill")
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>שאלה למחשבה:</b> איזה גרף הכי טוב כדי לראות את <b>מספר</b> המטופלים בכל קבוצה? איזה כדי להשוות <b>אחוזים</b> בין שתי הקבוצות כשהן שונות בגודלן (31 מול 9)? ואיזה מקשה להשוות דרגה אחת בין שתי הקבוצות?</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.25em; font-weight:bold; margin:0.6em 0 0.3em 0;">5. שונות משותפת (3): שני מספרים</p>
<p dir="rtl" style="direction:rtl; text-align:right;">האם מי שמתפקד טוב יותר בתנועה רצונית (<code dir="ltr">fma</code> גבוה) הוא גם עם ספסטיות קלה יותר (<code dir="ltr">tsrt</code> גבוה)? <b>א.</b> חשבו את מקדם המתאם של פירסון בין <code dir="ltr">fma</code> ל-<code dir="ltr">tsrt</code> ב-<code dir="ltr">sp2</code> ושמרו ב-<code dir="ltr">r</code> (עגלו לשתי ספרות).<br><b>ב.</b> ציירו גרף פיזור עם קו מגמה ישר. מקדם מתאם נע בין -1 ל-1: קרוב ל-0 = אין קשר ליניארי.</p>
<details dir="rtl" style="direction:rtl; text-align:right;"><summary><b>רמז</b></summary>
<pre dir="ltr" style="direction:ltr; text-align:left; background:#f4f4f4; padding:6px;"><code>r &lt;- round(cor(sp2$fma, sp2$tsrt), 2)
geom_point() + geom_smooth(method = "lm", formula = y ~ x)</code></pre>
</details>

In [ ]:
r <- round(cor(sp2$___, sp2$___), 2)                 # correlation of FMA-UE and TSRT, two decimals
r

In [ ]:
p_numnum <- ggplot(sp2, aes(x = fma, y = tsrt)) +
  geom_point(size = 3, alpha = 0.8) +                 # one point per patient
  geom_smooth(method = "___", formula = y ~ x) +      # "lm" = a straight trend line
  labs(x = "FMA-UE (0-66)", y = "TSRT (degrees)")
p_numnum

In [ ]:
stopifnot(abs(r - round(cor(sp2$fma, sp2$tsrt), 2)) < 1e-9,
          "GeomSmooth" %in% sapply(p_numnum$layers, function(l) class(l$geom)[1]))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>ג. שאלה למחשבה:</b> מה אומר המקדם שקיבלתם, ומה הוא <b>לא</b> אומר על מטופל בודד? (שני המבחנים מודדים דברים שונים: תפקוד מוטורי רצוני מול רפלקס מתיחה במנוחה.)</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.25em; font-weight:bold; margin:0.6em 0 0.3em 0;">6. השערה אחת</p>
<p dir="rtl" style="direction:rtl; text-align:right;">נסחו <b>השערה אחת</b> שעלתה מהגרפים והטבלאות: "אני מצפה ש... אצל מטופלים עם... כי...", ואיזה גרף הוביל אתכם אליה. זו השערה לבדיקה בנתונים חדשים ולא מסקנה: <b>גישוש אינו אישוש</b>. לחצו פעמיים על התא שמתחת כדי לערוך.</p>
<details dir="rtl" style="direction:rtl; text-align:right;"><summary><b>דוגמה למבנה</b></summary>
<p dir="rtl" style="direction:rtl; text-align:right;">"אני מצפה שאצל מטופלים עם שבץ המורגי הסף TSRT נמוך יותר (ספסטיות חמורה יותר) מאשר באיסכמי, כי בתרשים הקופסה החציון שלהם נמוך יותר. בקבוצה יש רק 9 מטופלים, ולכן צריך לבדוק זאת בנתונים נוספים."</p>
</details>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>ההשערה שלנו:</b> (כתבו כאן)</p>